# Week 9: CCE Ablation Study

This notebook validates that entropy spikes are a meaningful retrieval trigger:

1. **Same Setup** - Uses the exact same codebase and benchmark as Week 8
2. **CCE-Spike Retrieval** - Retrieve when CCE > 3.0 using query + confused tokens
3. **Ablation Baselines** - Random, Fixed-Interval, Query-Only, No-Retrieval
4. **Statistical Analysis** - Bootstrap CIs for rigorous comparison

---

## Setup

In [1]:
# Cell 1: Install dependencies
!pip install -q transformers torch accelerate scipy scikit-learn pandas matplotlib seaborn sentence-transformers

In [2]:
# Cell 2: Create directory structure
import os
import shutil

# Create module structure
os.makedirs('/content/orchestrator/entropy', exist_ok=True)
os.makedirs('/content/orchestrator/retrieval', exist_ok=True)
os.makedirs('/content/orchestrator/generation', exist_ok=True)
os.makedirs('/content/orchestrator/evaluation', exist_ok=True)

# Create root __init__.py
with open('/content/orchestrator/__init__.py', 'w') as f:
    f.write('"""Orchestrator package."""\n')

print("Directory structure created")

Directory structure created


In [3]:
# Cell 3: Upload module files
from google.colab import files

def upload_to_dir(target_dir):
    """Upload files and move to target directory."""
    uploaded = files.upload()
    for filename in uploaded.keys():
        if filename.endswith('.py'):
            dest = f'{target_dir}/{filename}'
            shutil.move(filename, dest)
            print(f"  OK {filename}")
    return uploaded

print("="*60)
print("STEP 1/4: Upload ENTROPY module files")
print("="*60)
print("\nUpload these files from packages/python-orchestrator/orchestrator/entropy/:")
print("  - __init__.py")
print("  - token_classifier.py  <-- REQUIRED for HybridClassifier")
print("  - (other entropy files as needed)")
upload_to_dir('/content/orchestrator/entropy')

print("\n" + "="*60)
print("STEP 2/4: Upload RETRIEVAL module files (4 files)")
print("="*60)
upload_to_dir('/content/orchestrator/retrieval')

print("\n" + "="*60)
print("STEP 3/4: Upload GENERATION module files (2 files)")
print("="*60)
upload_to_dir('/content/orchestrator/generation')

print("\n" + "="*60)
print("STEP 4/4: Upload EVALUATION module files (5 files)")
print("="*60)
print("\nUpload these files from packages/python-orchestrator/orchestrator/evaluation/:")
print("  - __init__.py")
print("  - benchmark.py")
print("  - benchmark_generator.py")
print("  - metrics.py")
print("  - runner.py")
upload_to_dir('/content/orchestrator/evaluation')

print("\n" + "="*60)
print("VERIFICATION")
print("="*60)
!ls -la /content/orchestrator/entropy/
!ls -la /content/orchestrator/evaluation/

STEP 1/4: Upload ENTROPY module files

Upload these files from packages/python-orchestrator/orchestrator/entropy/:
  - __init__.py
  - token_classifier.py  <-- REQUIRED for HybridClassifier
  - (other entropy files as needed)


Saving __init__.py to __init__.py
Saving calculator.py to calculator.py
Saving cce_computer.py to cce_computer.py
Saving measurement.py to measurement.py
Saving monitor.py to monitor.py
Saving spike_detector.py to spike_detector.py
Saving token_classifier.py to token_classifier.py
  OK __init__.py
  OK calculator.py
  OK cce_computer.py
  OK measurement.py
  OK monitor.py
  OK spike_detector.py
  OK token_classifier.py

STEP 2/4: Upload RETRIEVAL module files (4 files)


Saving __init__.py to __init__.py
Saving adaptive.py to adaptive.py
Saving context_manager.py to context_manager.py
Saving topic_inference.py to topic_inference.py
  OK __init__.py
  OK adaptive.py
  OK context_manager.py
  OK topic_inference.py

STEP 3/4: Upload GENERATION module files (2 files)


Saving __init__.py to __init__.py
Saving adaptive_generator.py to adaptive_generator.py
  OK __init__.py
  OK adaptive_generator.py

STEP 4/4: Upload EVALUATION module files (5 files)

Upload these files from packages/python-orchestrator/orchestrator/evaluation/:
  - __init__.py
  - benchmark.py
  - benchmark_generator.py
  - metrics.py
  - runner.py


Saving __init__.py to __init__.py
Saving benchmark.py to benchmark.py
Saving benchmark_generator.py to benchmark_generator.py
Saving metrics.py to metrics.py
Saving runner.py to runner.py
Saving stats.py to stats.py
  OK __init__.py
  OK benchmark.py
  OK benchmark_generator.py
  OK metrics.py
  OK runner.py
  OK stats.py

VERIFICATION
total 116
drwxr-xr-x 2 root root  4096 Jan 22 02:15 .
drwxr-xr-x 6 root root  4096 Jan 22 02:15 ..
-rw-r--r-- 1 root root  9942 Jan 22 02:15 calculator.py
-rw-r--r-- 1 root root  9362 Jan 22 02:15 cce_computer.py
-rw-r--r-- 1 root root  6854 Jan 22 02:15 __init__.py
-rw-r--r-- 1 root root 11385 Jan 22 02:15 measurement.py
-rw-r--r-- 1 root root 16903 Jan 22 02:15 monitor.py
-rw-r--r-- 1 root root 17802 Jan 22 02:15 spike_detector.py
-rw-r--r-- 1 root root 21519 Jan 22 02:15 token_classifier.py
total 172
drwxr-xr-x 2 root root  4096 Jan 22 02:16 .
drwxr-xr-x 6 root root  4096 Jan 22 02:15 ..
-rw-r--r-- 1 root root 65176 Jan 22 02:16 benchmark_generator.py

In [4]:
# Cell 4: Base imports
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Any
from collections import defaultdict
import json
import time
import gc

sys.path.insert(0, '/content')

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

import warnings
warnings.filterwarnings('ignore')

print("Base imports complete")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Base imports complete
PyTorch: 2.9.0+cu126
CUDA available: True


In [5]:
# Cell 5: Import evaluation modules
from orchestrator.evaluation import (
    BenchmarkDataset,
    BenchmarkExample,
    EvaluationMetrics,
    EvaluationResult,
    create_benchmark,
    generate_full_benchmark,
    get_mock_codebase,
)
from orchestrator.evaluation.benchmark import (
    Difficulty,
    Category,
)
from orchestrator.evaluation.runner import (
    BaselineRunner,
    ExperimentRunner,
    BaselineMethod,
)

print("All evaluation modules imported successfully!")
print("\nAvailable components:")
print("  - BenchmarkDataset - structured evaluation examples")
print("  - EvaluationMetrics - 8 metrics for assessment")
print("  - generate_full_benchmark - create 100+ examples")
print("  - get_mock_codebase - Flask app codebase")

All evaluation modules imported successfully!

Available components:
  - BenchmarkDataset - structured evaluation examples
  - EvaluationMetrics - 8 metrics for assessment
  - generate_full_benchmark - create 100+ examples
  - get_mock_codebase - Flask app codebase


In [10]:
# Cell 10: Initialize metrics
metrics = EvaluationMetrics(embedding_model='all-MiniLM-L6-v2')
print("EvaluationMetrics initialized with improved hallucination detection")

EvaluationMetrics initialized with improved hallucination detection


In [12]:
# Cell 12: Embedding Retriever
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

class EmbeddingRetriever:
    def __init__(self, documents: Dict[str, str]):
        self.documents = documents
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.doc_names = list(documents.keys())
        self.doc_contents = list(documents.values())
        self.doc_embeddings = self.model.encode(self.doc_contents)
        print(f"EmbeddingRetriever: {len(documents)} documents indexed")

    def retrieve(self, query: str, top_k: int = 2, deduplicate: bool = True) -> List[Dict]:
        query_emb = self.model.encode([query])
        sims = cosine_similarity(query_emb, self.doc_embeddings)[0]
        top_idx = np.argsort(sims)[-top_k:][::-1]
        return [{'source': self.doc_names[i], 'content': self.doc_contents[i], 'score': float(sims[i])} for i in top_idx]


print("EmbeddingRetriever class defined")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

EmbeddingRetriever: 14 documents indexed


In [13]:
# Cell 14: Token Classification & CCE Implementation (Using HybridClassifier)
from scipy.stats import entropy as scipy_entropy
from dataclasses import dataclass, field
from typing import Tuple
import random

# Import the advanced HybridClassifier from orchestrator
from orchestrator.entropy.token_classifier import HybridClassifier, KeywordClassifier

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

@dataclass
class MultiHopRetrievalResult:
    retrieved_files: List[str]
    retrieved_content: str
    scores: List[float]
    num_hops: int
    total_tokens: int
    trace: List[Dict[str, Any]]


class CCEQueryPlusTopKRetriever:
    """CCE Multi-Hop: Uses query + confused CODE tokens for retrieval.

    Uses HybridClassifier (keyword + embedding fallback) for token classification.
    """

    def __init__(self, base_retriever, tokenizer, model,
                 top_k: int = 2, max_retrievals: int = 5,
                 uncertainty_threshold: float = 3.0,
                 top_k_tokens: int = 10,
                 max_gen_tokens: int = 200,
                 cooldown_tokens: int = 5,
                 file_list_context: str = "",
                 use_hybrid_classifier: bool = True):
        self.retriever = base_retriever
        self.tokenizer = tokenizer
        self.model = model
        self.top_k = top_k
        self.max_retrievals = max_retrievals
        self.uncertainty_threshold = uncertainty_threshold
        self.top_k_tokens = top_k_tokens
        self.max_gen_tokens = max_gen_tokens
        self.cooldown_tokens = cooldown_tokens
        self.file_list_context = file_list_context
        self.use_hybrid_classifier = use_hybrid_classifier

        # Initialize classifier
        if use_hybrid_classifier:
            print("Using HybridClassifier (keyword + embedding fallback)")
            self.classifier = HybridClassifier(
                embedding_model='all-MiniLM-L6-v2',
                embedding_margin=0.05,
                use_embedding_cache=True
            )
        else:
            print("Using KeywordClassifier (keyword-only)")
            self.classifier = KeywordClassifier()

        # Build vocabulary classification (one-time)
        self._build_vocab_classification()

    def _build_vocab_classification(self):
        """Classify all tokens in vocabulary using HybridClassifier."""
        vocab_size = len(self.tokenizer)
        self.code_indices = []
        self.language_indices = []
        self.other_indices = []

        print(f"Classifying {vocab_size} tokens...")
        for token_id in range(vocab_size):
            try:
                token = self.tokenizer.decode([token_id]).strip()
                if not token:
                    self.other_indices.append(token_id)
                    continue

                # Use classifier (hybrid or keyword-only)
                result = self.classifier.classify(token)

                if result == 'code':
                    self.code_indices.append(token_id)
                elif result == 'language':
                    self.language_indices.append(token_id)
                else:
                    self.other_indices.append(token_id)
            except:
                self.other_indices.append(token_id)

        self.code_indices = np.array(self.code_indices)
        self.language_indices = np.array(self.language_indices)

        print(f"Vocab classification complete:")
        print(f"  Code tokens: {len(self.code_indices)}")
        print(f"  Language tokens: {len(self.language_indices)}")
        print(f"  Other tokens: {len(self.other_indices)}")

        # Show classifier stats if hybrid
        if self.use_hybrid_classifier and hasattr(self.classifier, 'get_stats'):
            stats = self.classifier.get_stats()
            total = stats['keyword_hits'] + stats['embedding_hits'] + stats['other']
            if total > 0:
                print(f"  Keyword hits: {stats['keyword_hits']} ({100*stats['keyword_hits']/total:.1f}%)")
                print(f"  Embedding hits: {stats['embedding_hits']} ({100*stats['embedding_hits']/total:.1f}%)")

    def _compute_cce(self, logits: torch.Tensor) -> Tuple[float, float, float]:
        logits_np = logits.cpu().numpy()

        if len(self.code_indices) > 0:
            code_logits = logits_np[self.code_indices]
            code_logits_stable = code_logits - np.max(code_logits)
            code_probs = np.exp(code_logits_stable) / np.sum(np.exp(code_logits_stable))
            h_code = float(scipy_entropy(code_probs, base=2))
        else:
            h_code = 0.0

        if len(self.language_indices) > 0:
            lang_logits = logits_np[self.language_indices]
            lang_logits_stable = lang_logits - np.max(lang_logits)
            lang_probs = np.exp(lang_logits_stable) / np.sum(np.exp(lang_logits_stable))
            h_lang = float(scipy_entropy(lang_probs, base=2))
        else:
            h_lang = 0.0

        return h_code - h_lang, h_code, h_lang

    def _extract_code_tokens_from_logits(self, logits: torch.Tensor) -> List[str]:
        logits_np = logits.cpu().numpy()
        code_logits = logits_np[self.code_indices]
        top_within_code = np.argsort(code_logits)[-self.top_k_tokens:][::-1]

        code_tokens = []
        for i in top_within_code:
            token_id = self.code_indices[i]
            token = self.tokenizer.decode([token_id]).strip()
            if len(token) > 1:
                code_tokens.append(token)
        return code_tokens

    def retrieve(self, query: str) -> MultiHopRetrievalResult:
        if self.file_list_context:
            prompt = f"{query}\n\n{self.file_list_context}\n\n"
        else:
            prompt = f"{query}\n\n"

        retrieved_files = []
        retrieved_content = []
        all_scores = []
        trace = []
        seen_files = set()
        retrieval_count = 0
        last_retrieval_pos = -100

        inputs = self.tokenizer(prompt, return_tensors='pt').to(self.model.device)
        generated_ids = inputs['input_ids']

        for i in range(self.max_gen_tokens):
            with torch.no_grad():
                outputs = self.model(generated_ids)
                logits = outputs.logits[0, -1, :]

            del outputs  # Free memory
            cce, h_code, h_lang = self._compute_cce(logits)

            if i < 3:
                print(f"    Token {i}: CCE={cce:.3f} (H_code={h_code:.2f}, H_lang={h_lang:.2f})")

            tokens_since_last = i - last_retrieval_pos
            in_cooldown = tokens_since_last < self.cooldown_tokens

            if cce > self.uncertainty_threshold and not in_cooldown and retrieval_count < self.max_retrievals:
                confused_tokens = self._extract_code_tokens_from_logits(logits)
                retrieval_query = f"{query} {' '.join(confused_tokens)}"

                results = self.retriever.retrieve(retrieval_query, top_k=self.top_k, deduplicate=False)
                new_files = [r for r in results if r['source'] not in seen_files]

                if new_files:
                    for r in new_files:
                        seen_files.add(r['source'])
                        retrieved_files.append(r['source'])
                        retrieved_content.append(r['content'])
                        all_scores.append(r['score'])

                    new_context = "\n\n".join([r['content'] for r in new_files])
                    context_text = f"\n\nRelevant context:\n{new_context}\n\n"
                    context_ids = self.tokenizer.encode(context_text, return_tensors='pt').to(self.model.device)
                    generated_ids = torch.cat([generated_ids, context_ids], dim=-1)

                trace.append({
                    'hop': retrieval_count + 1,
                    'position': i,
                    'cce': cce,
                    'confused_tokens': confused_tokens[:5],
                    'new_files': [r['source'] for r in new_files] if new_files else [],
                })

                retrieval_count += 1
                last_retrieval_pos = i
                print(f"    SPIKE {retrieval_count} at {i}: CCE={cce:.2f}, tokens={confused_tokens[:5]}")

            next_token = torch.argmax(logits).unsqueeze(0).unsqueeze(0)
            generated_ids = torch.cat([generated_ids, next_token.to(self.model.device)], dim=-1)

            # Periodic memory cleanup
            if i % 50 == 0 and i > 0:
                torch.cuda.empty_cache()

            if next_token.item() == self.tokenizer.eos_token_id:
                break

        if not trace:
            trace.append({'hop': 0, 'method': 'no_spike_detected'})

        content = "\n\n".join(retrieved_content)
        print(f"    Total: {retrieval_count} retrievals, files: {retrieved_files}")

        return MultiHopRetrievalResult(
            retrieved_files=retrieved_files,
            retrieved_content=content,
            scores=all_scores,
            num_hops=retrieval_count,
            total_tokens=len(self.tokenizer.encode(content)) if content else 0,
            trace=trace
        )

print("CCEQueryPlusTopKRetriever defined (with HybridClassifier support)")

CCEQueryPlusTopKRetriever defined (with HybridClassifier support)


In [14]:
# Cell 13: Load LLM model
MODEL_NAME = "Qwen/Qwen2.5-Coder-0.5B-Instruct"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded. Vocab size: {len(tokenizer)}")

Loading Qwen/Qwen2.5-Coder-0.5B-Instruct...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded. Vocab size: 151665
File list tokens: 108


In [15]:
# Cell 15: Ablation Baselines

class RandomRetrievalBaseline:
    def __init__(self, retriever, tokenizer, model, max_gen_tokens=200, file_list_context=""):
        self.retriever = retriever
        self.tokenizer = tokenizer
        self.model = model
        self.max_gen_tokens = max_gen_tokens
        self.file_list_context = file_list_context

    def retrieve(self, query: str, num_retrievals: int = 2, seed: int = None) -> MultiHopRetrievalResult:
        if seed is not None:
            random.seed(seed)

        positions = sorted(random.sample(range(10, self.max_gen_tokens - 10), min(num_retrievals, self.max_gen_tokens - 20)))

        prompt = f"{query}\n\n{self.file_list_context}\n\n" if self.file_list_context else f"{query}\n\n"
        inputs = self.tokenizer(prompt, return_tensors='pt').to(self.model.device)
        generated_ids = inputs['input_ids']

        retrieved_files, retrieved_content, all_scores, trace = [], [], [], []
        seen_files = set()
        retrieval_count = 0

        for i in range(self.max_gen_tokens):
            if i in positions:
                gen_text = self.tokenizer.decode(generated_ids[0], skip_special_tokens=True)
                search_query = f"{query} {gen_text[-100:]}"
                results = self.retriever.retrieve(search_query, top_k=2)

                for r in results:
                    if r['source'] not in seen_files:
                        seen_files.add(r['source'])
                        retrieved_files.append(r['source'])
                        retrieved_content.append(r['content'])
                        all_scores.append(r['score'])

                if results:
                    new_context = "\n\n".join([r['content'] for r in results])
                    context_ids = self.tokenizer.encode(f"\n\nRelevant context:\n{new_context}\n\n", return_tensors='pt').to(self.model.device)
                    generated_ids = torch.cat([generated_ids, context_ids], dim=-1)

                trace.append({'position': i, 'method': 'random'})
                retrieval_count += 1

            with torch.no_grad():
                outputs = self.model(generated_ids)
                logits = outputs.logits[0, -1, :]

            next_token = torch.argmax(logits).unsqueeze(0).unsqueeze(0)
            del outputs  # Free memory
            generated_ids = torch.cat([generated_ids, next_token.to(self.model.device)], dim=-1)

            # Periodic memory cleanup
            if i % 50 == 0 and i > 0:
                torch.cuda.empty_cache()

            if next_token.item() == self.tokenizer.eos_token_id:
                break

        content = "\n\n".join(retrieved_content)
        return MultiHopRetrievalResult(retrieved_files, content, all_scores, retrieval_count,
                                       len(self.tokenizer.encode(content)) if content else 0, trace)


class FixedIntervalBaseline:
    def __init__(self, retriever, tokenizer, model, max_gen_tokens=200, file_list_context=""):
        self.retriever = retriever
        self.tokenizer = tokenizer
        self.model = model
        self.max_gen_tokens = max_gen_tokens
        self.file_list_context = file_list_context

    def retrieve(self, query: str, interval: int = 50, max_retrievals: int = 2) -> MultiHopRetrievalResult:
        prompt = f"{query}\n\n{self.file_list_context}\n\n" if self.file_list_context else f"{query}\n\n"
        inputs = self.tokenizer(prompt, return_tensors='pt').to(self.model.device)
        generated_ids = inputs['input_ids']

        retrieved_files, retrieved_content, all_scores, trace = [], [], [], []
        seen_files = set()
        retrieval_count = 0

        for i in range(self.max_gen_tokens):
            if i > 0 and i % interval == 0 and retrieval_count < max_retrievals:
                gen_text = self.tokenizer.decode(generated_ids[0], skip_special_tokens=True)
                results = self.retriever.retrieve(f"{query} {gen_text[-100:]}", top_k=2)

                for r in results:
                    if r['source'] not in seen_files:
                        seen_files.add(r['source'])
                        retrieved_files.append(r['source'])
                        retrieved_content.append(r['content'])
                        all_scores.append(r['score'])

                if results:
                    new_context = "\n\n".join([r['content'] for r in results])
                    context_ids = self.tokenizer.encode(f"\n\nRelevant context:\n{new_context}\n\n", return_tensors='pt').to(self.model.device)
                    generated_ids = torch.cat([generated_ids, context_ids], dim=-1)

                trace.append({'position': i, 'method': 'fixed_interval'})
                retrieval_count += 1

            with torch.no_grad():
                outputs = self.model(generated_ids)
                logits = outputs.logits[0, -1, :]

            next_token = torch.argmax(logits).unsqueeze(0).unsqueeze(0)
            del outputs  # Free memory
            generated_ids = torch.cat([generated_ids, next_token.to(self.model.device)], dim=-1)

            # Periodic memory cleanup
            if i % 50 == 0 and i > 0:
                torch.cuda.empty_cache()

            # Periodic memory cleanup
            if i % 50 == 0 and i > 0:
                torch.cuda.empty_cache()

            if next_token.item() == self.tokenizer.eos_token_id:
                break

        content = "\n\n".join(retrieved_content)
        return MultiHopRetrievalResult(retrieved_files, content, all_scores, retrieval_count,
                                       len(self.tokenizer.encode(content)) if content else 0, trace)


class QueryOnlyBaseline:
    def __init__(self, retriever, tokenizer, model, max_gen_tokens=200):
        self.retriever = retriever
        self.tokenizer = tokenizer
        self.model = model
        self.max_gen_tokens = max_gen_tokens

    def generate(self, query: str) -> Tuple[str, List[str]]:
        results = self.retriever.retrieve(query, top_k=2)
        context = "\n\n".join([r['content'] for r in results])
        files = [r['source'] for r in results]

        prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
        inputs = self.tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(inputs['input_ids'], max_new_tokens=self.max_gen_tokens,
                                          pad_token_id=self.tokenizer.eos_token_id)
        answer = self.tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        return answer, files


class NoRetrievalBaseline:
    def __init__(self, tokenizer, model, max_gen_tokens=200):
        self.tokenizer = tokenizer
        self.model = model
        self.max_gen_tokens = max_gen_tokens

    def generate(self, query: str) -> str:
        prompt = f"Question: {query}\nAnswer:"
        inputs = self.tokenizer(prompt, return_tensors='pt').to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(inputs['input_ids'], max_new_tokens=self.max_gen_tokens,
                                          pad_token_id=self.tokenizer.eos_token_id)
        return self.tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

print("Baselines defined")

Baselines defined


In [22]:
# Cell 11: Generate with CCE trace function

def generate_with_cce_trace(query: str, max_tokens: int = 100) -> Dict:
    """Generate answer WITHOUT retrieval, logging CCE at each token position.
    
    Uses global code_token_mask and lang_token_mask created in Cell C.
    """
    
    prompt = "Question: " + query + "\n" + "Answer:"
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(model.device)
    
    tokens_generated = []
    cce_trace = []
    h_code_trace = []
    h_lang_trace = []
    
    current_ids = input_ids
    
    for step in range(max_tokens):
        with torch.no_grad():
            outputs = model(current_ids)
            logits = outputs.logits[0, -1, :]
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            
            # Use global code/language token masks
            code_probs = probs[code_token_mask]
            lang_probs = probs[lang_token_mask]
            
            # Compute entropies
            if code_probs.sum() > 0:
                code_probs_norm = code_probs / code_probs.sum()
                h_code = scipy_entropy(code_probs_norm, base=2)
            else:
                h_code = 0.0
                
            if lang_probs.sum() > 0:
                lang_probs_norm = lang_probs / lang_probs.sum()
                h_lang = scipy_entropy(lang_probs_norm, base=2)
            else:
                h_lang = 0.0
            
            cce = h_code - h_lang
            
            cce_trace.append(cce)
            h_code_trace.append(h_code)
            h_lang_trace.append(h_lang)
            
            # Sample next token
            next_token_id = torch.argmax(logits).item()
            next_token = tokenizer.decode([next_token_id])
            tokens_generated.append(next_token)
            
            # Check for EOS
            if next_token_id == tokenizer.eos_token_id:
                break
                
            current_ids = torch.cat([current_ids, torch.tensor([[next_token_id]]).to(model.device)], dim=1)
    
    full_answer = ''.join(tokens_generated)
    
    return {
        'query': query,
        'answer': full_answer,
        'tokens': tokens_generated,
        'cce_trace': cce_trace,
        'h_code_trace': h_code_trace,
        'h_lang_trace': h_lang_trace,
        'spike_positions': [i for i, cce in enumerate(cce_trace) if cce > 3.0],
    }

print("generate_with_cce_trace function defined (uses global token masks)")


SPIKE-ERROR CORRELATION EXPERIMENT
Generating WITHOUT retrieval to capture CCE traces...

[1/6] arch_001: What is the overall structure of the Fla...
  Generated 100 tokens
  Spikes at positions: [6, 16, 23, 26, 33, 56, 70]

[2/6] arch_002: How does the authentication system work ...
  Generated 97 tokens
  Spikes at positions: [7, 8, 9, 10, 13, 27, 37, 52, 96]

[3/6] api_001: How do I register a new user via the API...
  Generated 100 tokens
  Spikes at positions: [11, 28, 43, 51, 52, 67, 77, 84, 89]

[4/6] api_002: How do I authenticate and make protected...
  Generated 100 tokens
  Spikes at positions: [8, 10, 24, 25, 35, 91]

[5/6] impl_001: How is JWT token creation implemented?...
  Generated 100 tokens
  Spikes at positions: [6, 69, 87]

[6/6] impl_002: How is password hashing implemented?...
  Generated 60 tokens
  Spikes at positions: [45, 55]


Collected 6 examples with CCE traces


In [23]:
# Cell 12: Compute hallucination trace function

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

hallu_embed_model = SentenceTransformer('all-MiniLM-L6-v2')

def compute_hallucination_trace(tokens: List[str], ground_truth: str) -> List[Dict]:
    """
    Compute hallucination score at each token position using semantic distance.
    
    As generation progresses toward ground truth, similarity should INCREASE.
    A DROP in similarity indicates hallucination.
    """
    gt_embedding = hallu_embed_model.encode([ground_truth])[0]
    
    trace = []
    cumulative_text = ""
    prev_similarity = 0.0
    
    for i, token in enumerate(tokens):
        cumulative_text += token
        
        # Embed cumulative generated text
        gen_embedding = hallu_embed_model.encode([cumulative_text])[0]
        
        # Compute similarity to ground truth
        similarity = float(cosine_similarity([gen_embedding], [gt_embedding])[0][0])
        
        # Hallucination score = 1 - similarity (higher = more hallucinated)
        hallucination_score = 1.0 - similarity
        
        # Detect similarity drop (potential hallucination point)
        similarity_drop = prev_similarity - similarity if i > 0 else 0.0
        
        trace.append({
            'position': i,
            'token': token,
            'cumulative_similarity': similarity,
            'hallucination_score': hallucination_score,
            'similarity_drop': similarity_drop,
            'is_drop': similarity_drop > 0.01,
        })
        
        prev_similarity = similarity
    
    return trace

print("compute_hallucination_trace function defined")


Computing hallucination traces via semantic distance...
  arch_001: 6 similarity drops detected
  arch_002: 15 similarity drops detected
  api_001: 6 similarity drops detected
  api_002: 7 similarity drops detected
  impl_001: 9 similarity drops detected
  impl_002: 9 similarity drops detected

Hallucination traces computed for all examples


---
## Cerberus Experiment: Real Codebase Validation

Using **Cerberus** (Python validation library) as a real-world codebase to test CCE spike-error correlation.

In [ ]:
# Cell A: Clone and Load Cerberus Codebase
import os
import subprocess

print("="*70)
print("STEP 1: Clone Cerberus Repository")
print("="*70)

# Clone if not exists
if not os.path.exists('cerberus'):
    subprocess.run(['git', 'clone', 'https://github.com/pyeve/cerberus.git', '--depth', '1'], check=True)
    print("Cloned cerberus repository")
else:
    print("Cerberus already exists, skipping clone")

# Load codebase into dict format
def load_cerberus_codebase():
    """Load Cerberus source files into dict format for retriever."""
    files = {}
    cerberus_src = 'cerberus/cerberus'
    
    for root, dirs, filenames in os.walk(cerberus_src):
        for f in filenames:
            if f.endswith('.py'):
                path = os.path.join(root, f)
                rel_path = path.replace('cerberus/', '')
                try:
                    with open(path, 'r', encoding='utf-8') as fp:
                        files[rel_path] = fp.read()
                except Exception as e:
                    print(f"  Warning: Could not read {path}: {e}")
    
    return files

cerberus_codebase = load_cerberus_codebase()

print(f"\nCerberus Codebase Loaded")
print("="*70)
print(f"Total files: {len(cerberus_codebase)}")
print(f"Total size: {sum(len(v) for v in cerberus_codebase.values()):,} characters")
print("\nFiles:")
for path, content in sorted(cerberus_codebase.items()):
    print(f"  {path} ({len(content):,} chars)")

# Create file list context for Cerberus
cerberus_file_list = "Available files in Cerberus codebase:\n"
for path in sorted(cerberus_codebase.keys()):
    cerberus_file_list += f"- {path}\n"
cerberus_file_list += "\nWhen answering questions about Cerberus, refer to the relevant files above."
print(f"\nFile list context created ({len(cerberus_file_list)} chars)")

In [ ]:
# Cell B: Generate 50 Benchmark Examples for Cerberus
# These examples REQUIRE reading the actual Cerberus source code to answer correctly

CERBERUS_BENCHMARK = [
    # === API USAGE (10 examples) ===
    {
        'id': 'cerb_api_001',
        'category': 'api_usage',
        'difficulty': 'easy',
        'query': 'How do I create a basic Cerberus validator and validate a document?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['Validator', 'validate', 'schema', 'document', 'errors'],
        'ground_truth_answer': 'Create a Validator instance with a schema dict, then call validate(document) method.'
    },
    {
        'id': 'cerb_api_002',
        'category': 'api_usage',
        'difficulty': 'easy',
        'query': 'What method returns the validation errors after calling validate()?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['errors', 'property', 'dict'],
        'ground_truth_answer': 'The errors property returns a dict of validation errors.'
    },
    {
        'id': 'cerb_api_003',
        'category': 'api_usage',
        'difficulty': 'medium',
        'query': 'How do I define a custom validation rule in Cerberus?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['_validate', 'def', 'constraint', 'field', 'value'],
        'ground_truth_answer': 'Define a method named _validate_<rulename>(self, constraint, field, value) in a Validator subclass.'
    },
    {
        'id': 'cerb_api_004',
        'category': 'api_usage',
        'difficulty': 'medium',
        'query': 'What is the schema rule to make a field required in Cerberus?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['required', 'True', 'schema'],
        'ground_truth_answer': "Use 'required': True in the field schema."
    },
    {
        'id': 'cerb_api_005',
        'category': 'api_usage',
        'difficulty': 'medium',
        'query': 'How do I normalize a value before validation using coercion?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['coerce', 'normalize', 'int', 'str'],
        'ground_truth_answer': "Use 'coerce': callable in the schema to transform the value before validation."
    },
    {
        'id': 'cerb_api_006',
        'category': 'api_usage',
        'difficulty': 'easy',
        'query': 'What types can be specified in the type rule?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['type', 'string', 'integer', 'float', 'list', 'dict', 'boolean'],
        'ground_truth_answer': 'Common types include string, integer, float, boolean, list, dict, and more.'
    },
    {
        'id': 'cerb_api_007',
        'category': 'api_usage',
        'difficulty': 'medium',
        'query': 'How do I validate nested documents (subdocuments) in Cerberus?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['schema', 'nested', 'type', 'dict'],
        'ground_truth_answer': "Use 'type': 'dict' with 'schema': {...} to validate nested documents."
    },
    {
        'id': 'cerb_api_008',
        'category': 'api_usage',
        'difficulty': 'hard',
        'query': 'How do I use dependencies to require a field only when another field has a specific value?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['dependencies', 'field', 'value'],
        'ground_truth_answer': "Use 'dependencies': {'other_field': 'value'} to make field required conditionally."
    },
    {
        'id': 'cerb_api_009',
        'category': 'api_usage',
        'difficulty': 'medium',
        'query': 'What is the allowed rule used for in Cerberus?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['allowed', 'list', 'values', 'enum'],
        'ground_truth_answer': "The 'allowed' rule restricts values to a list of allowed values (like an enum)."
    },
    {
        'id': 'cerb_api_010',
        'category': 'api_usage',
        'difficulty': 'hard',
        'query': 'How do I validate a list of items where each item must match a schema?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['type', 'list', 'schema', 'items'],
        'ground_truth_answer': "Use 'type': 'list' with 'schema': {...} to validate each item in the list."
    },
    
    # === IMPLEMENTATION (15 examples) ===
    {
        'id': 'cerb_impl_001',
        'category': 'implementation',
        'difficulty': 'hard',
        'query': 'What method does Validator call internally to normalize a value using coercion?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['_normalize_coerce', 'coerce', 'normalize'],
        'ground_truth_answer': 'The _normalize_coerce method handles coercion during normalization.'
    },
    {
        'id': 'cerb_impl_002',
        'category': 'implementation',
        'difficulty': 'hard',
        'query': 'What is the naming convention for custom validation rule methods?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['_validate_', 'prefix', 'method'],
        'ground_truth_answer': 'Custom validators are methods prefixed with _validate_ followed by the rule name.'
    },
    {
        'id': 'cerb_impl_003',
        'category': 'implementation',
        'difficulty': 'hard',
        'query': 'What attribute stores the current document being validated?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['document', 'self.document', '_document'],
        'ground_truth_answer': 'The document attribute or _document stores the current document being validated.'
    },
    {
        'id': 'cerb_impl_004',
        'category': 'implementation',
        'difficulty': 'hard',
        'query': 'How does Cerberus internally store the current field path during nested validation?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['document_path', 'field', 'path'],
        'ground_truth_answer': 'The document_path attribute tracks the current field path during nested validation.'
    },
    {
        'id': 'cerb_impl_005',
        'category': 'implementation',
        'difficulty': 'hard',
        'query': 'What method is called to validate a single field against all its rules?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['_validate', 'field', 'rules'],
        'ground_truth_answer': 'The __validate_definition or similar method validates a field against all rules.'
    },
    {
        'id': 'cerb_impl_006',
        'category': 'implementation',
        'difficulty': 'hard',
        'query': 'What class is used to represent validation errors in Cerberus?',
        'ground_truth_files': ['cerberus/errors.py'],
        'keywords': ['ValidationError', 'ErrorDefinition', 'errors'],
        'ground_truth_answer': 'ValidationError or ErrorDefinition class represents validation errors.'
    },
    {
        'id': 'cerb_impl_007',
        'category': 'implementation',
        'difficulty': 'hard',
        'query': 'How does Cerberus determine if a validator method exists for a rule?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['getattr', 'hasattr', '_validate_', 'method'],
        'ground_truth_answer': 'Uses getattr/hasattr to check for _validate_<rule> method existence.'
    },
    {
        'id': 'cerb_impl_008',
        'category': 'implementation',
        'difficulty': 'hard',
        'query': 'What method handles the normalization phase before validation?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['normalize', '_normalize', 'normalization'],
        'ground_truth_answer': 'The _normalize or normalize method handles normalization before validation.'
    },
    {
        'id': 'cerb_impl_009',
        'category': 'implementation',
        'difficulty': 'hard',
        'query': 'How are schema rules registered and discovered in Cerberus?',
        'ground_truth_files': ['cerberus/schema.py', 'cerberus/validator.py'],
        'keywords': ['rules', 'schema', 'register', 'definition'],
        'ground_truth_answer': 'Schema rules are discovered via _validate_ method introspection and schema registry.'
    },
    {
        'id': 'cerb_impl_010',
        'category': 'implementation',
        'difficulty': 'hard',
        'query': 'What happens when validation fails and errors need to be recorded?',
        'ground_truth_files': ['cerberus/validator.py', 'cerberus/errors.py'],
        'keywords': ['_error', 'errors', 'add', 'append'],
        'ground_truth_answer': 'The _error method is called to record validation failures in the errors dict.'
    },
    {
        'id': 'cerb_impl_011',
        'category': 'implementation',
        'difficulty': 'hard',
        'query': 'How does Cerberus handle unknown fields in a document?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['allow_unknown', 'unknown', 'purge_unknown'],
        'ground_truth_answer': 'The allow_unknown and purge_unknown settings control unknown field handling.'
    },
    {
        'id': 'cerb_impl_012',
        'category': 'implementation',
        'difficulty': 'hard',
        'query': 'What is the order of operations during validation (normalize, then validate)?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['normalize', 'validate', 'order', 'sequence'],
        'ground_truth_answer': 'Normalization (coerce, default, rename) happens before validation rules are applied.'
    },
    {
        'id': 'cerb_impl_013',
        'category': 'implementation',
        'difficulty': 'hard',
        'query': 'How does the type rule validator determine if a value matches the expected type?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['_validate_type', 'isinstance', 'type', 'types'],
        'ground_truth_answer': 'The _validate_type method uses type checking/isinstance to verify value type.'
    },
    {
        'id': 'cerb_impl_014',
        'category': 'implementation',
        'difficulty': 'hard',
        'query': 'What internal structure stores the validated/normalized document?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['document', 'normalized', 'result'],
        'ground_truth_answer': 'The document property returns the normalized/validated document after processing.'
    },
    {
        'id': 'cerb_impl_015',
        'category': 'implementation',
        'difficulty': 'hard',
        'query': 'How does Cerberus support schema inheritance and extension?',
        'ground_truth_files': ['cerberus/schema.py', 'cerberus/validator.py'],
        'keywords': ['extend', 'inherit', 'schema', 'merge'],
        'ground_truth_answer': 'Schemas can be extended by merging dicts or using schema inheritance patterns.'
    },
    
    # === ERROR HANDLING (10 examples) ===
    {
        'id': 'cerb_err_001',
        'category': 'error_handling',
        'difficulty': 'medium',
        'query': 'What exception is raised when the schema itself is invalid?',
        'ground_truth_files': ['cerberus/errors.py', 'cerberus/schema.py'],
        'keywords': ['SchemaError', 'invalid', 'schema'],
        'ground_truth_answer': 'SchemaError is raised when the schema definition is invalid.'
    },
    {
        'id': 'cerb_err_002',
        'category': 'error_handling',
        'difficulty': 'medium',
        'query': 'How do I get a human-readable error message from validation errors?',
        'ground_truth_files': ['cerberus/errors.py', 'cerberus/validator.py'],
        'keywords': ['errors', 'message', 'str'],
        'ground_truth_answer': 'Access validator.errors dict which contains field-keyed error messages.'
    },
    {
        'id': 'cerb_err_003',
        'category': 'error_handling',
        'difficulty': 'hard',
        'query': 'What error code is used for type mismatch errors?',
        'ground_truth_files': ['cerberus/errors.py'],
        'keywords': ['BAD_TYPE', 'TYPE', 'error_code'],
        'ground_truth_answer': 'The BAD_TYPE or similar error code indicates type mismatch.'
    },
    {
        'id': 'cerb_err_004',
        'category': 'error_handling',
        'difficulty': 'hard',
        'query': 'How are nested document errors represented in the errors structure?',
        'ground_truth_files': ['cerberus/errors.py', 'cerberus/validator.py'],
        'keywords': ['nested', 'errors', 'dict', 'path'],
        'ground_truth_answer': 'Nested errors are represented as nested dicts mirroring the document structure.'
    },
    {
        'id': 'cerb_err_005',
        'category': 'error_handling',
        'difficulty': 'medium',
        'query': 'What error is returned when a required field is missing?',
        'ground_truth_files': ['cerberus/errors.py', 'cerberus/validator.py'],
        'keywords': ['REQUIRED_FIELD', 'required', 'missing'],
        'ground_truth_answer': 'REQUIRED_FIELD or similar error code for missing required fields.'
    },
    {
        'id': 'cerb_err_006',
        'category': 'error_handling',
        'difficulty': 'hard',
        'query': 'How do I customize error messages for specific rules?',
        'ground_truth_files': ['cerberus/errors.py', 'cerberus/validator.py'],
        'keywords': ['error', 'message', 'custom', 'define'],
        'ground_truth_answer': 'Override error messages by subclassing or using error handler customization.'
    },
    {
        'id': 'cerb_err_007',
        'category': 'error_handling',
        'difficulty': 'hard',
        'query': 'What class handles error message formatting and interpolation?',
        'ground_truth_files': ['cerberus/errors.py'],
        'keywords': ['BasicErrorHandler', 'ErrorHandler', 'format'],
        'ground_truth_answer': 'BasicErrorHandler or ErrorHandler class handles error formatting.'
    },
    {
        'id': 'cerb_err_008',
        'category': 'error_handling',
        'difficulty': 'medium',
        'query': 'What error is returned when a value is not in the allowed list?',
        'ground_truth_files': ['cerberus/errors.py'],
        'keywords': ['UNALLOWED_VALUE', 'allowed', 'value'],
        'ground_truth_answer': 'UNALLOWED_VALUE error code for values not in allowed list.'
    },
    {
        'id': 'cerb_err_009',
        'category': 'error_handling',
        'difficulty': 'hard',
        'query': 'How does Cerberus accumulate multiple errors for the same field?',
        'ground_truth_files': ['cerberus/errors.py', 'cerberus/validator.py'],
        'keywords': ['errors', 'append', 'list', 'accumulate'],
        'ground_truth_answer': 'Errors are accumulated as lists per field, allowing multiple errors per field.'
    },
    {
        'id': 'cerb_err_010',
        'category': 'error_handling',
        'difficulty': 'hard',
        'query': 'What is the ErrorDefinition class used for in Cerberus?',
        'ground_truth_files': ['cerberus/errors.py'],
        'keywords': ['ErrorDefinition', 'code', 'rule'],
        'ground_truth_answer': 'ErrorDefinition defines error types with codes and message templates.'
    },
    
    # === SCHEMA RULES (10 examples) ===
    {
        'id': 'cerb_schema_001',
        'category': 'schema_rules',
        'difficulty': 'medium',
        'query': 'How does the minlength rule work for strings and lists?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['minlength', 'len', 'minimum'],
        'ground_truth_answer': 'minlength checks that string/list length is >= specified value.'
    },
    {
        'id': 'cerb_schema_002',
        'category': 'schema_rules',
        'difficulty': 'medium',
        'query': 'What is the difference between min and minlength rules?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['min', 'minlength', 'value', 'length'],
        'ground_truth_answer': 'min compares numeric value, minlength compares length of sequences.'
    },
    {
        'id': 'cerb_schema_003',
        'category': 'schema_rules',
        'difficulty': 'hard',
        'query': 'How does the regex rule validate string patterns?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['regex', 're', 'pattern', 'match'],
        'ground_truth_answer': 'regex rule uses re module to match string against provided pattern.'
    },
    {
        'id': 'cerb_schema_004',
        'category': 'schema_rules',
        'difficulty': 'hard',
        'query': 'What does the nullable rule do?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['nullable', 'None', 'null'],
        'ground_truth_answer': 'nullable: True allows None as a valid value for the field.'
    },
    {
        'id': 'cerb_schema_005',
        'category': 'schema_rules',
        'difficulty': 'hard',
        'query': 'How does the default rule work during normalization?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['default', 'normalize', 'missing'],
        'ground_truth_answer': 'default provides a value when field is missing during normalization.'
    },
    {
        'id': 'cerb_schema_006',
        'category': 'schema_rules',
        'difficulty': 'hard',
        'query': 'What is the rename rule used for?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['rename', 'field', 'key'],
        'ground_truth_answer': 'rename changes the field key during normalization.'
    },
    {
        'id': 'cerb_schema_007',
        'category': 'schema_rules',
        'difficulty': 'hard',
        'query': 'How does the anyof rule work for complex validation?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['anyof', 'oneof', 'allof', 'noneof'],
        'ground_truth_answer': 'anyof validates that value matches at least one of the provided schemas.'
    },
    {
        'id': 'cerb_schema_008',
        'category': 'schema_rules',
        'difficulty': 'hard',
        'query': 'What is the keysrules rule used for in dict validation?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['keysrules', 'keys', 'dict'],
        'ground_truth_answer': 'keysrules applies validation rules to dictionary keys.'
    },
    {
        'id': 'cerb_schema_009',
        'category': 'schema_rules',
        'difficulty': 'hard',
        'query': 'How does valuesrules differ from schema for dict validation?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['valuesrules', 'values', 'schema', 'dict'],
        'ground_truth_answer': 'valuesrules applies same rules to all dict values, schema defines per-key rules.'
    },
    {
        'id': 'cerb_schema_010',
        'category': 'schema_rules',
        'difficulty': 'hard',
        'query': 'What does the empty rule control?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['empty', 'allow', 'string'],
        'ground_truth_answer': 'empty: False disallows empty strings/lists as valid values.'
    },
    
    # === EDGE CASES (5 examples) ===
    {
        'id': 'cerb_edge_001',
        'category': 'edge_cases',
        'difficulty': 'hard',
        'query': 'What happens when you validate an empty document {}?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['empty', 'document', 'required'],
        'ground_truth_answer': 'Empty document passes unless schema has required fields.'
    },
    {
        'id': 'cerb_edge_002',
        'category': 'edge_cases',
        'difficulty': 'hard',
        'query': 'How does Cerberus handle circular references in nested schemas?',
        'ground_truth_files': ['cerberus/validator.py', 'cerberus/schema.py'],
        'keywords': ['circular', 'recursive', 'reference'],
        'ground_truth_answer': 'Circular schemas may cause recursion issues; use string references or lazy loading.'
    },
    {
        'id': 'cerb_edge_003',
        'category': 'edge_cases',
        'difficulty': 'hard',
        'query': 'What happens when coercion raises an exception?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['coerce', 'exception', 'error'],
        'ground_truth_answer': 'Coercion exceptions are caught and recorded as validation errors.'
    },
    {
        'id': 'cerb_edge_004',
        'category': 'edge_cases',
        'difficulty': 'hard',
        'query': 'Can you use the same Validator instance for multiple documents?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['validate', 'multiple', 'reuse', 'clear'],
        'ground_truth_answer': 'Yes, each validate() call resets state for a new document.'
    },
    {
        'id': 'cerb_edge_005',
        'category': 'edge_cases',
        'difficulty': 'hard',
        'query': 'What happens when a rule validator method signature is incorrect?',
        'ground_truth_files': ['cerberus/validator.py'],
        'keywords': ['signature', 'method', 'error', 'TypeError'],
        'ground_truth_answer': 'Incorrect signatures may cause TypeError or be silently ignored.'
    },
]

# Convert to BenchmarkExample objects
from dataclasses import dataclass
from typing import List

@dataclass
class CerberusExample:
    id: str
    category: str
    difficulty: str
    query: str
    ground_truth_files: List[str]
    keywords: List[str]
    ground_truth_answer: str

cerberus_examples = [CerberusExample(**ex) for ex in CERBERUS_BENCHMARK]

print("="*70)
print("CERBERUS BENCHMARK DATASET")
print("="*70)
print(f"Total examples: {len(cerberus_examples)}")
print(f"\nBy Category:")
from collections import Counter
cat_counts = Counter(ex.category for ex in cerberus_examples)
for cat, count in sorted(cat_counts.items()):
    print(f"  {cat}: {count}")
print(f"\nBy Difficulty:")
diff_counts = Counter(ex.difficulty for ex in cerberus_examples)
for diff, count in sorted(diff_counts.items()):
    print(f"  {diff}: {count}")

In [ ]:
# Cell C: Initialize Retrievers with Cerberus Codebase

print("="*70)
print("INITIALIZING RETRIEVERS FOR CERBERUS")
print("="*70)

# Create embedding retriever with Cerberus codebase
cerberus_retriever = EmbeddingRetriever(cerberus_codebase)
print(f"Embedding retriever initialized with {len(cerberus_codebase)} files")

# Create CCE retriever for Cerberus
print("Initializing CCE Retriever for Cerberus (threshold=3.0)...")
cerberus_cce_retriever = CCEQueryPlusTopKRetriever(
    base_retriever=cerberus_retriever,
    tokenizer=tokenizer,
    model=model,
    uncertainty_threshold=3.0,
    max_gen_tokens=200,
    file_list_context=cerberus_file_list
)
print("CCE retriever initialized!")

# Create global boolean masks for token classification (used by generate_with_cce_trace)
vocab_size = len(tokenizer)
code_token_mask = np.zeros(vocab_size, dtype=bool)
lang_token_mask = np.zeros(vocab_size, dtype=bool)
code_token_mask[cerberus_cce_retriever.code_indices] = True
lang_token_mask[cerberus_cce_retriever.language_indices] = True
print(f"Token masks created: {code_token_mask.sum()} code, {lang_token_mask.sum()} lang")

# Create baseline retrievers
cerberus_random_baseline = RandomRetrievalBaseline(
    cerberus_retriever, tokenizer, model, 200, cerberus_file_list
)
cerberus_fixed_baseline = FixedIntervalBaseline(
    cerberus_retriever, tokenizer, model, 200, cerberus_file_list
)

print("All Cerberus retrievers initialized:")
print("  - cerberus_cce_retriever (CCE spike-triggered)")
print("  - cerberus_random_baseline (random timing)")
print("  - cerberus_fixed_baseline (fixed interval)")

# Compute baseline tokens for Cerberus
sep = chr(10) + chr(10)
cerberus_full_context = sep.join([f"# {path}" + chr(10) + content for path, content in cerberus_codebase.items()])
cerberus_baseline_tokens = len(tokenizer.encode(cerberus_full_context))
print(f"Cerberus baseline tokens (full context): {cerberus_baseline_tokens:,}")


In [ ]:
# Cell D: Run Ablation Study with Cerberus Codebase

NUM_CERBERUS_EXAMPLES = 20  # Sample for faster iteration
NUM_RANDOM_SEEDS = 1  # Reduced from 3 for memory stability

print("="*70)
print("CERBERUS ABLATION STUDY")
print("="*70)
print(f"Examples: {NUM_CERBERUS_EXAMPLES}")
print(f"Random seeds: {NUM_RANDOM_SEEDS}")
print("="*70)

def cleanup_memory():
    """Free GPU memory between iterations."""
    gc.collect()
    torch.cuda.empty_cache()

# Sample examples stratified by category
import random
random.seed(42)

sampled_cerberus = []
categories = list(set(ex.category for ex in cerberus_examples))
per_cat = NUM_CERBERUS_EXAMPLES // len(categories)

for cat in categories:
    cat_examples = [ex for ex in cerberus_examples if ex.category == cat]
    sampled_cerberus.extend(random.sample(cat_examples, min(per_cat, len(cat_examples))))

# Fill remaining if needed
remaining = NUM_CERBERUS_EXAMPLES - len(sampled_cerberus)
if remaining > 0:
    unused = [ex for ex in cerberus_examples if ex not in sampled_cerberus]
    sampled_cerberus.extend(random.sample(unused, min(remaining, len(unused))))

print(f"Sampled {len(sampled_cerberus)} examples")

# Storage for results
cerberus_results = {
    'cce_spike': [],
    'random': [],
    'fixed': [],
    'query_only': [],
    'no_retrieval': []
}

def evaluate_cerberus(example, answer: str, retrieved_files: list, tokens_used: int, 
                      method_name: str, num_retrievals: int = 0) -> EvaluationResult:
    """Evaluate a generated answer against Cerberus ground truth."""
    return metrics.evaluate(
        example_id=example.id,
        generated_answer=answer,
        ground_truth_answer=example.ground_truth_answer,
        retrieved_files=retrieved_files,
        ground_truth_files=example.ground_truth_files,
        ground_truth_keywords=example.keywords,
        tokens_used=tokens_used,
        baseline_tokens=cerberus_baseline_tokens,
        num_retrievals=num_retrievals,
        method=method_name
    )

# Run evaluation
for idx, ex in enumerate(sampled_cerberus):
    print(f"[{idx+1}/{len(sampled_cerberus)}] {ex.id}: {ex.query[:50]}...")
    
    # === CCE-Spike ===
    cleanup_memory()  # Clear before retrieval
    cce_result = cerberus_cce_retriever.retrieve(ex.query)
    if cce_result.num_hops > 0 and cce_result.retrieved_content:
        prompt = f"Context:{chr(10)}{cce_result.retrieved_content}{chr(10)}{chr(10)}Question: {ex.query}{chr(10)}Answer:"
    else:
        prompt = f"Question: {ex.query}{chr(10)}Answer:"
    
    cce_input = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        cce_output = model.generate(**cce_input, max_new_tokens=150, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    cce_answer = tokenizer.decode(cce_output[0][cce_input['input_ids'].shape[1]:], skip_special_tokens=True)
    
    del cce_output, cce_input
    cleanup_memory()
    
    cce_eval = evaluate_cerberus(ex, cce_answer, cce_result.retrieved_files, 
                                  len(tokenizer.encode(prompt)), 'cce_spike', cce_result.num_hops)
    cerberus_results['cce_spike'].append(cce_eval)
    print(f"  CCE-Spike: retrievals={cce_result.num_hops}, correct={cce_eval.answer_correctness:.3f}")
    
    target_ret = max(1, cce_result.num_hops)
    
    # === Random (multiple seeds) ===
    for seed in range(NUM_RANDOM_SEEDS):
        cleanup_memory()  # Clear before retrieval
        rand_result = cerberus_random_baseline.retrieve(ex.query, num_retrievals=target_ret, seed=seed+idx*100)
        if rand_result.num_hops > 0 and rand_result.retrieved_content:
            prompt = f"Context:{chr(10)}{rand_result.retrieved_content}{chr(10)}{chr(10)}Question: {ex.query}{chr(10)}Answer:"
        else:
            prompt = f"Question: {ex.query}{chr(10)}Answer:"
        rand_input = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            rand_output = model.generate(**rand_input, max_new_tokens=150, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        rand_answer = tokenizer.decode(rand_output[0][rand_input['input_ids'].shape[1]:], skip_special_tokens=True)
        del rand_output, rand_input
        cleanup_memory()
        
        rand_eval = evaluate_cerberus(ex, rand_answer, rand_result.retrieved_files,
                                       len(tokenizer.encode(prompt)), 'random', rand_result.num_hops)
        cerberus_results['random'].append(rand_eval)
    
    # === Fixed Interval ===
    cleanup_memory()  # Clear before retrieval
    interval = max(30, 200 // target_ret)
    fixed_result = cerberus_fixed_baseline.retrieve(ex.query, interval=interval, max_retrievals=target_ret)
    if fixed_result.num_hops > 0 and fixed_result.retrieved_content:
        prompt = f"Context:{chr(10)}{fixed_result.retrieved_content}{chr(10)}{chr(10)}Question: {ex.query}{chr(10)}Answer:"
    else:
        prompt = f"Question: {ex.query}{chr(10)}Answer:"
    fixed_input = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        fixed_output = model.generate(**fixed_input, max_new_tokens=150, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    fixed_answer = tokenizer.decode(fixed_output[0][fixed_input['input_ids'].shape[1]:], skip_special_tokens=True)
    del fixed_output, fixed_input
    cleanup_memory()
    
    fixed_eval = evaluate_cerberus(ex, fixed_answer, fixed_result.retrieved_files,
                                    len(tokenizer.encode(prompt)), 'fixed', fixed_result.num_hops)
    cerberus_results['fixed'].append(fixed_eval)
    
    # === Query-Only ===
    qo_prompt = f"{cerberus_file_list}{chr(10)}{chr(10)}Question: {ex.query}{chr(10)}Answer:"
    qo_input = tokenizer(qo_prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        qo_output = model.generate(**qo_input, max_new_tokens=150, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    qo_answer = tokenizer.decode(qo_output[0][qo_input['input_ids'].shape[1]:], skip_special_tokens=True)
    del qo_output, qo_input
    cleanup_memory()
    
    qo_eval = evaluate_cerberus(ex, qo_answer, [], len(tokenizer.encode(qo_prompt)), 'query_only', 0)
    cerberus_results['query_only'].append(qo_eval)
    
    # === No Retrieval ===
    nr_prompt = f"Question: {ex.query}{chr(10)}Answer:"
    nr_input = tokenizer(nr_prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        nr_output = model.generate(**nr_input, max_new_tokens=150, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    nr_answer = tokenizer.decode(nr_output[0][nr_input['input_ids'].shape[1]:], skip_special_tokens=True)
    del nr_output, nr_input
    cleanup_memory()
    
    nr_eval = evaluate_cerberus(ex, nr_answer, [], len(tokenizer.encode(nr_prompt)), 'no_retrieval', 0)
    cerberus_results['no_retrieval'].append(nr_eval)

print("="*70)
print("CERBERUS ABLATION COMPLETE")
print("="*70)


In [ ]:
# Cell E: Spike-Error Correlation with Cerberus

print("="*70)
print("CERBERUS: SPIKE-ERROR CORRELATION EXPERIMENT")
print("="*70)
print("Generating WITHOUT retrieval to capture CCE traces...")

# Use subset of hard examples for spike-error analysis
hard_examples = [ex for ex in sampled_cerberus if ex.difficulty == 'hard'][:10]
if len(hard_examples) < 6:
    hard_examples = sampled_cerberus[:10]  # Fallback

print(f"\nAnalyzing {len(hard_examples)} examples for spike-error correlation")

cerberus_spike_data = []

for idx, ex in enumerate(hard_examples):
    print(f"\n[{idx+1}/{len(hard_examples)}] {ex.id}: {ex.query[:40]}...")
    
    # Generate without retrieval, logging CCE
    trace_result = generate_with_cce_trace(ex.query, max_tokens=100)
    
    print(f"  Generated {len(trace_result['tokens'])} tokens")
    print(f"  Spikes at positions: {trace_result['spike_positions']}")
    
    cerberus_spike_data.append({
        'example_id': ex.id,
        'query': ex.query,
        'ground_truth': ex.ground_truth_answer,
        'tokens': trace_result['tokens'],
        'cce_trace': trace_result['cce_trace'],
        'spike_positions': trace_result['spike_positions'],
    })

print(f"\nCollected {len(cerberus_spike_data)} examples with CCE traces")

# Compute hallucination traces
print("\nComputing hallucination traces via semantic distance...")

for data in cerberus_spike_data:
    hallu_trace = compute_hallucination_trace(data['tokens'], data['ground_truth'])
    data['hallu_trace'] = hallu_trace
    data['hallu_scores'] = [h['hallucination_score'] for h in hallu_trace]
    data['drop_positions'] = [i for i, h in enumerate(hallu_trace) if h.get('is_drop', False)]
    print(f"  {data['example_id']}: {len(data['drop_positions'])} similarity drops detected")

print("\nHallucination traces computed for Cerberus examples")

# Correlation Analysis
print("\n" + "="*70)
print("CERBERUS: CORRELATION ANALYSIS")
print("="*70)

cerb_all_cce = []
cerb_all_hallu = []
cerb_all_spike_flags = []
cerb_all_drop_flags = []

for data in cerberus_spike_data:
    cce_trace = data['cce_trace']
    hallu_scores = data['hallu_scores']
    min_len = min(len(cce_trace), len(hallu_scores))
    
    for i in range(min_len):
        cerb_all_cce.append(cce_trace[i])
        cerb_all_hallu.append(hallu_scores[i])
        cerb_all_spike_flags.append(1 if cce_trace[i] > 3.0 else 0)
        cerb_all_drop_flags.append(1 if i in data['drop_positions'] else 0)

cerb_all_cce = np.array(cerb_all_cce)
cerb_all_hallu = np.array(cerb_all_hallu)
cerb_all_spike_flags = np.array(cerb_all_spike_flags)
cerb_all_drop_flags = np.array(cerb_all_drop_flags)

print(f"\nTotal token positions analyzed: {len(cerb_all_cce)}")
print(f"Total spikes (CCE > 3.0): {sum(cerb_all_spike_flags)}")
print(f"Total similarity drops: {sum(cerb_all_drop_flags)}")

# Pearson correlation
if len(cerb_all_cce) > 2:
    cerb_r_pearson, cerb_p_pearson = pearsonr(cerb_all_cce, cerb_all_hallu)
    cerb_r_spearman, cerb_p_spearman = spearmanr(cerb_all_cce, cerb_all_hallu)
    print(f"\n--- Direct Correlation ---")
    print(f"Pearson:  r = {cerb_r_pearson:.4f}, p = {cerb_p_pearson:.4f}")
    print(f"Spearman: r = {cerb_r_spearman:.4f}, p = {cerb_p_spearman:.4f}")
else:
    cerb_r_pearson, cerb_p_pearson = 0, 1
    print("\nNot enough data for correlation")

# Time-lagged correlation
print(f"\n--- Time-Lagged Correlation ---")
cerb_lagged_results = []
for lag in [1, 2, 3, 5, 10]:
    if len(cerb_all_cce) > lag + 2:
        cce_shifted = cerb_all_cce[:-lag]
        hallu_shifted = cerb_all_hallu[lag:]
        r_lag, p_lag = pearsonr(cce_shifted, hallu_shifted)
        cerb_lagged_results.append({'lag': lag, 'r': r_lag, 'p': p_lag})
        print(f"  Lag={lag}: r = {r_lag:.4f}, p = {p_lag:.4f}")

# Conditional probability
print(f"\n--- Conditional Probability ---")
cerb_spike_and_drop = sum((cerb_all_spike_flags == 1) & (cerb_all_drop_flags == 1))
cerb_spike_no_drop = sum((cerb_all_spike_flags == 1) & (cerb_all_drop_flags == 0))
cerb_no_spike_and_drop = sum((cerb_all_spike_flags == 0) & (cerb_all_drop_flags == 1))
cerb_no_spike_no_drop = sum((cerb_all_spike_flags == 0) & (cerb_all_drop_flags == 0))

cerb_total_spikes = cerb_spike_and_drop + cerb_spike_no_drop
cerb_total_no_spikes = cerb_no_spike_and_drop + cerb_no_spike_no_drop

cerb_p_drop_spike = cerb_spike_and_drop / cerb_total_spikes if cerb_total_spikes > 0 else 0
cerb_p_drop_no_spike = cerb_no_spike_and_drop / cerb_total_no_spikes if cerb_total_no_spikes > 0 else 0

print(f"P(drop | spike)    = {cerb_p_drop_spike:.4f} ({cerb_spike_and_drop}/{cerb_total_spikes})")
print(f"P(drop | no_spike) = {cerb_p_drop_no_spike:.4f} ({cerb_no_spike_and_drop}/{cerb_total_no_spikes})")

cerb_relative_risk = cerb_p_drop_spike / cerb_p_drop_no_spike if cerb_p_drop_no_spike > 0 else float('inf')
print(f"Relative Risk = {cerb_relative_risk:.2f}x")

# Chi-square test
print(f"\n--- Chi-Square Test ---")
cerb_contingency = np.array([
    [cerb_spike_and_drop, cerb_spike_no_drop],
    [cerb_no_spike_and_drop, cerb_no_spike_no_drop]
])
if cerb_contingency.min() > 0:
    cerb_chi2, cerb_p_chi2, _, _ = chi2_contingency(cerb_contingency)
    print(f"Chi-square = {cerb_chi2:.4f}, p = {cerb_p_chi2:.4f}")
else:
    cerb_chi2, cerb_p_chi2 = 0, 1
    print("Cannot compute chi-square (zero cells)")

# Final Verdict for Cerberus
print("\n" + "="*70)
print("CERBERUS: FINAL VERDICT")
print("="*70)

cerb_criteria_met = 0
if abs(cerb_r_pearson) > 0.3:
    print(f"✓ Correlation r={cerb_r_pearson:.3f} > 0.3")
    cerb_criteria_met += 1
else:
    print(f"✗ Correlation r={cerb_r_pearson:.3f} < 0.3")

if cerb_relative_risk > 1.5:
    print(f"✓ Relative Risk = {cerb_relative_risk:.2f}x > 1.5x")
    cerb_criteria_met += 1
else:
    print(f"✗ Relative Risk = {cerb_relative_risk:.2f}x < 1.5x")

if cerb_p_chi2 < 0.05:
    print(f"✓ Chi-square p={cerb_p_chi2:.4f} < 0.05")
    cerb_criteria_met += 1
else:
    print(f"✗ Chi-square p={cerb_p_chi2:.4f} >= 0.05")

print(f"\nCriteria met: {cerb_criteria_met}/3")

if cerb_criteria_met >= 2:
    cerb_spike_verdict = "SPIKES PREDICT HALLUCINATIONS"
elif cerb_criteria_met == 1:
    cerb_spike_verdict = "WEAK EVIDENCE"
else:
    cerb_spike_verdict = "SPIKES DO NOT PREDICT HALLUCINATIONS"

print(f"\nVERDICT: {cerb_spike_verdict}")

In [ ]:
# Cell F: Export Final Results

print("="*70)
print("CERBERUS EXPERIMENT RESULTS")
print("="*70)

# Compute aggregate metrics
cerb_methods = ['cce_spike', 'random', 'fixed', 'query_only', 'no_retrieval']
cerb_agg = {}

for method in cerb_methods:
    results_list = cerberus_results[method]
    if results_list:
        cerb_agg[method] = {
            'correctness': np.mean([r.answer_correctness for r in results_list]),
            'hallucination': np.mean([r.hallucination_rate for r in results_list]),
            'composite': np.mean([r.get_composite_score() for r in results_list]),
            'n': len(results_list)
        }

print("")
print("=== ABLATION RESULTS ===")
print(f"{'Method':<15} {'Correct':>10} {'Hallu':>10} {'Composite':>10}")
print("-" * 50)
for method in cerb_methods:
    if method in cerb_agg:
        m = cerb_agg[method]
        print(f"{method:<15} {m['correctness']:>10.3f} {m['hallucination']:>10.3f} {m['composite']:>10.3f}")

# CCE vs Random
cce_comp = cerb_agg['cce_spike']['composite']
rand_comp = cerb_agg['random']['composite']
diff = cce_comp - rand_comp

print("")
print("=== CCE vs RANDOM ===")
print(f"CCE Composite:    {cce_comp:.3f}")
print(f"Random Composite: {rand_comp:.3f}")
print(f"Difference:       {diff:+.3f}")

if diff > 0.05:
    ablation_verdict = "CCE OUTPERFORMS RANDOM"
elif diff < -0.05:
    ablation_verdict = "RANDOM OUTPERFORMS CCE"
else:
    ablation_verdict = "NO SIGNIFICANT DIFFERENCE"

print(f"Verdict: {ablation_verdict}")

print("")
print("=== SPIKE-ERROR CORRELATION ===")
print(f"Pearson r:     {cerb_r_pearson:.4f}")
print(f"Relative Risk: {cerb_relative_risk:.2f}x")
print(f"Chi-square p:  {cerb_p_chi2:.4f}")
print(f"Criteria met:  {cerb_criteria_met}/3")
print(f"Verdict: {cerb_spike_verdict}")

# Export
export_data = {
    'experiment': 'cerberus_cce_ablation',
    'codebase': {
        'name': 'Cerberus',
        'files': len(cerberus_codebase),
        'chars': sum(len(v) for v in cerberus_codebase.values()),
    },
    'ablation': cerb_agg,
    'spike_error': {
        'pearson_r': float(cerb_r_pearson),
        'relative_risk': float(cerb_relative_risk) if cerb_relative_risk != float('inf') else 999,
        'chi2_p': float(cerb_p_chi2),
        'criteria_met': cerb_criteria_met,
        'verdict': cerb_spike_verdict,
    },
    'overall_verdict': ablation_verdict,
}

with open('cerberus_results.json', 'w') as f:
    json.dump(export_data, f, indent=2)

print("")
print("Exported: cerberus_results.json")
